# Phase 2 — Post-Training Quantization (PTQ)

**Goal:** baseline both PTQ INT8 and PTQ INT4. PTQ is the cheapest method (no training, just calibrate min/max per tensor and round) — every QAT variant in notebooks 03-05 has to beat it on KL divergence to justify its training cost.

**Inputs:**
- `/kaggle/input/sqat-baseline/results/baseline/fp32_logits.pt` from notebook 01.

**Outputs:**
- `results/ptq_int8/training_results.json`
- `results/ptq_int4/training_results.json`
- `models/checkpoints/ptq_int{4,8}/` — PTQ state dicts for GGUF export.

**Estimated time:** ~30 min total (no training; just calibration + perplexity eval per bit-width).

## 1. Setup

In [ ]:
import os, sys, shutil, subprocess

WORKING_DIR  = "/kaggle/working"
REPO_DIR     = f"{WORKING_DIR}/scheduled-qat-for-slm"
GITHUB_URL   = "https://github.com/JpCurada/scheduled-qat-for-slm.git"
REPO_DATASET = "/kaggle/input/sqat-repo"
BASELINE_DIR = "/kaggle/input/sqat-baseline/results/baseline"   # from notebook 01

if not os.path.exists(REPO_DIR):
    if os.path.exists(REPO_DATASET):
        shutil.copytree(REPO_DATASET, REPO_DIR)
    else:
        subprocess.run(["git", "clone", "--depth", "1", GITHUB_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

In [ ]:
!pip install -q -e ".[viz]"

In [ ]:
from pathlib import Path
import torch

FP32_LOGITS = Path(BASELINE_DIR) / "fp32_logits.pt"
assert FP32_LOGITS.exists(), (
    f"FP32 logits not found at {FP32_LOGITS}.\n"
    "Add the `sqat-baseline` Kaggle dataset (output of notebook 01) as input to this notebook."
)

size_gb = FP32_LOGITS.stat().st_size / 1e9
print(f"FP32 logits OK: {FP32_LOGITS}  ({size_gb:.2f} GB)")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")

## 2. Configuration

PTQ has no training, but we still want the perplexity eval to use a reduced sequence length (512) so KL divergence is computed against logits from notebook 01 at the matching length. We patch the YAML configs in-place and write Kaggle copies under `configs_kaggle/`.

In [ ]:
import yaml

MODEL_CACHE = f"{WORKING_DIR}/models/base"
CHECKPOINT_DIR = f"{WORKING_DIR}/models/checkpoints"
RESULTS_DIR = f"{WORKING_DIR}/results"
KAGGLE_CFG = Path(REPO_DIR) / "configs_kaggle" / "ptq"
KAGGLE_CFG.mkdir(parents=True, exist_ok=True)

PTQ_SEQ_LEN = 512   # match LOGIT_SEQ_LEN from notebook 01

def make_kaggle_ptq_config(src: str, dst: Path) -> Path:
    with open(src) as f:
        cfg = yaml.safe_load(f)
    cfg["model"]["cache_dir"] = MODEL_CACHE
    cfg["calibration"]["seq_length"] = PTQ_SEQ_LEN
    cfg["calibration"]["num_samples"] = 64    # smaller calib for Kaggle
    cfg["export"]["output_dir"] = f"{WORKING_DIR}/models/gguf/"
    with dst.open("w") as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    return dst

ptq8_cfg = make_kaggle_ptq_config("configs/ptq/ptq_int8.yaml", KAGGLE_CFG / "ptq_int8.yaml")
ptq4_cfg = make_kaggle_ptq_config("configs/ptq/ptq_int4.yaml", KAGGLE_CFG / "ptq_int4.yaml")

print("INT8 config:", ptq8_cfg)
print("INT4 config:", ptq4_cfg)
!cat {ptq8_cfg}

## 3. PTQ INT8

Calibrates min/max from 64 train sequences, quantizes Linear weights to INT8, and evaluates perplexity + KL divergence vs FP32 logits.

In [ ]:
from src.training.trainer import run_experiment
import torch, gc

results_int8 = run_experiment(
    config_path=str(ptq8_cfg),
    device_str="cuda",
    baseline_logits=str(FP32_LOGITS),
    run_benchmarks=False,
)

print("\nPTQ INT8 results:")
for k, v in results_int8.items():
    print(f"  {k:20s}  {v}")

gc.collect(); torch.cuda.empty_cache()

## 4. PTQ INT4

Same calibration, INT4 weights. Expect a noticeable PPL bump and meaningful KL divergence — that's the gap QAT methods will try to close.

In [ ]:
results_int4 = run_experiment(
    config_path=str(ptq4_cfg),
    device_str="cuda",
    baseline_logits=str(FP32_LOGITS),
    run_benchmarks=False,
)

print("\nPTQ INT4 results:")
for k, v in results_int4.items():
    print(f"  {k:20s}  {v}")

gc.collect(); torch.cuda.empty_cache()

## 5. Comparison table

FP32 baseline numbers come from notebook 01's saved JSON; INT8/INT4 from this notebook.

In [ ]:
import json
import pandas as pd

with open(Path(BASELINE_DIR) / "baseline_results.json") as f:
    fp32 = json.load(f)

rows = [
    {"method": "FP32 baseline", "bits": 32,
     "perplexity": fp32.get("perplexity"), "kl_divergence": 0.0,
     "size_gb": 6.5},
    {"method": "PTQ", "bits": 8,
     "perplexity": results_int8.get("perplexity"),
     "kl_divergence": results_int8.get("kl_divergence"),
     "size_gb": 1.7},
    {"method": "PTQ", "bits": 4,
     "perplexity": results_int4.get("perplexity"),
     "kl_divergence": results_int4.get("kl_divergence"),
     "size_gb": 0.85},
]
df = pd.DataFrame(rows)
df["ppl_delta_pct"] = ((df["perplexity"] / df["perplexity"].iloc[0]) - 1.0) * 100
df.round(4)

In [ ]:
summary_path = Path(RESULTS_DIR) / "ptq_summary.json"
summary_path.parent.mkdir(parents=True, exist_ok=True)
summary = {"int8": results_int8, "int4": results_int4, "fp32": fp32}
with summary_path.open("w") as f:
    json.dump(summary, f, indent=2, default=str)
print(f"Wrote {summary_path}")

## Next steps

Save outputs as a Kaggle dataset (e.g. `sqat-ptq`) so notebook 07 can load all PTQ numbers for the cross-method comparison.

Proceed to `03_standard_qat.ipynb`.